In [17]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pypsa

BASE = Path("/Users/fowfloi/Documents/pypsa-earth")
RES  = BASE / "results"
OUT  = BASE / "analysis" / "paper1" / "out"
OUT.mkdir(parents=True, exist_ok=True)

USD_PER_EUR = 1.3277
HOURS_PER_YEAR = 8760

RUNS = [
    "nigeria_2030_wacc_0700","nigeria_2030_wacc_0900",
    "nigeria_2030_wacc_1100","nigeria_2030_wacc_1200",
    "nigeria_2030_wacc_1300","nigeria_2030_wacc_1379",
    "nigeria_2030_wacc_1468","nigeria_2030_wacc_1600",
    "nigeria_2030_wacc_1800","nigeria_2030_wacc_2000",
    "nigeria_2030_wacc_2300","nigeria_2030_wacc_2700",
    "nigeria_2030_wacc_0700_gas_600",
    "nigeria_2030_wacc_1379_gas_600",
    "nigeria_2030_wacc_2000_gas_600",
    "cap_exp_2030_uniform_wacc",
    "cap_exp_2030_split_wacc",
]

WACC_BY_TAG = {
    "0700":0.07,"0900":0.09,"1100":0.11,"1200":0.12,"1300":0.13,
    "1379":0.1379,"1468":0.1468,"1600":0.16,"1800":0.18,
    "2000":0.20,"2300":0.23,"2700":0.27,
}

def find_network(run):
    p = RES / run / "networks"
    if not p.exists(): return None
    hits = list(p.glob("elec_s_*_lcopt_*.nc"))
    return hits[0] if hits else None

def tag_of(run):
    for t in WACC_BY_TAG:
        if run.endswith(f"_{t}") or run.endswith(f"_{t}_gas_600"):
            return t
    if "uniform" in run: return "uniform"
    if "split" in run: return "split"
    return run

def wacc_of(run):
    t = tag_of(run)
    return WACC_BY_TAG.get(t, np.nan)

def gas_price_of(run):
    return 6.00 if "gas_600" in run else 3.39